# Pipeline Stage 1 — từ raw đến `stage1_panel.parquet`

Notebook này chạy **đúng** pipeline đang có trong `scripts/`, theo đúng thứ tự,
và giải thích mỗi bước làm gì. Nó không chép lại code của pipeline: mỗi cell gọi
thẳng script thật qua tiến trình con và stream log ra ngay dưới cell, nên cái bạn
đọc ở đây với cái chạy ra `data/final/stage1_panel.parquet` là một.

**Đích đến:** `data/final/stage1_panel.parquet` — 932.204 dòng × 209 cột (178 MB).
Một dòng là một **năm quan sát của một quan hệ xuất khẩu** (nước nhập khẩu ×
nhóm sản phẩm), phục vụ phân tích sinh tồn: quan hệ đó sống được bao lâu.

| | |
|---|---:|
| Nước xuất khẩu | Việt Nam |
| Nước nhập khẩu | 147 |
| Nhóm sản phẩm (family) | 4.365 xuất hiện / 4.522 trong khoá |
| Spell (quan hệ) | 189.830 |
| Sự kiện "chết" đã xác nhận | 112.885 |
| Cửa sổ thời gian | 2002–2025 |

### Folder này tự chạy được một mình

`notebook/` **là gốc dự án**, không phải một thư mục con của cái gì khác. Mọi
script trong `scripts/` mở đầu bằng cùng một dòng:

```python
HERE = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
```

— tức là chúng coi *thư mục cha của `scripts/`* là gốc rồi đọc/ghi `<gốc>/data/`,
`<gốc>/selection/`, `<gốc>/docs/`, `<gốc>/.env`. Vì `scripts/` nằm ngay trong
folder này, gốc đó **chính là folder này**, và notebook với script nhìn cùng một
tập file. Không có chuyện "notebook đọc một đằng, script ghi một nẻo".

```
notebook/                   ← gốc dự án
├── stage1_pipeline.ipynb   ← bạn đang ở đây
├── nbtools/                công cụ dùng chung, để cell notebook ngắn và đọc được
├── scripts/                toàn bộ code của pipeline (30 script)
├── selection/              input làm bằng tay — không script nào sinh ra được
├── docs/                   từ điển biến, kế hoạch sửa lỗi, báo cáo kiểm định
├── data/
│   ├── raw/                tải từ API/nguồn gốc  (765 MB — đi kèm sẵn)
│   ├── interim/            bảng trung gian       (pipeline dựng ra)
│   └── final/              stage1_panel.parquet  (pipeline dựng ra)
├── logs/                   log đầy đủ của từng bước
├── requirements.txt
└── README.md               cài đặt, sự cố thường gặp, cách gửi đi tiếp
```

### 32 bước, 6 phần

| Phần | Bước | Việc |
|---|---|---|
| 0 | 1–3 | Chuẩn bị: môi trường, input tay, bảng tra cứu mã HS |
| 1 | 4–13 | **Thu thập raw** — 10 nguồn, mỗi nguồn một cell vì chúng hỏng độc lập nhau |
| 2 | 14–23 | **Tiền xử lý** — dựng khoá sản phẩm, spell, và từng module đặc trưng |
| 3 | 24–26 | **Merge thành data tổng** → `data/interim/panel_final.csv` |
| 4 | 27–29 | **Dựng panel Stage 1** → `data/final/stage1_panel.parquet` |
| 5 | 30–32 | **Kiểm định** 18 phép thử, và bàn giao |

### Chạy thế nào

Có sẵn `data/raw/` thì **Run All là xong** — Phần 1 tự bỏ qua sạch vì mọi file
đã có, và notebook dựng lại từ bước 12. **Khoảng 16 phút**, không cần mạng,
không cần API key — đo ngày 22/09/2026 trên máy 8 GB WSL2, và panel dựng ra
trùng bản v2 tham chiếu ở mọi cột. Bước 1 ngay dưới đây sẽ nói chính xác máy bạn đang ở tình trạng nào.

---
## Phần 0 — Chuẩn bị

### Bước 1. Kiểm tra môi trường và dữ liệu sẵn có

Cell này không dựng gì cả. Nó trả lời ba câu trước khi bạn bấm Run All.

**Một: folder có đủ không.** `scripts/`, `selection/` và `data/` phải nằm ngay
trong folder này. Nếu bạn nhận được bản gửi kèm thì chúng đã có sẵn; nếu thiếu,
notebook dừng ở đây thay vì để bạn phát hiện ở bước 14.

**Hai: môi trường có chạy được không** — đang dùng python nào, đủ gói chưa.
Dòng `polars` phải hiện **1.38.1**: đó là bản đã dựng ra panel v2, và phép kiểm
A18 (hai lần dựng phải cho file giống hệt nhau) so sánh tới từng cột.

**Ba: dữ liệu đang có tới đâu, và vì thế notebook này chạy mất bao lâu.** Bảng
"Dữ liệu sẵn có" đi theo đúng thứ tự pipeline, nên nhìn dòng đầu tiên còn thiếu
là biết notebook sẽ bắt đầu làm việc thật từ chỗ nào.

In [ ]:
import pathlib
import sys


# Gốc dự án = folder notebook/, nhận ra bằng nbtools/config.py nằm ngay trong nó.
# Dò từ thư mục làm việc đi ngược lên, và ở mỗi mức thử cả thư mục con
# `notebook/`: JupyterLab đặt thư mục làm việc ngay tại file .ipynb, còn VS Code
# thường đặt ở gốc workspace — cả hai trường hợp đều phải chạy được.
def _find_root() -> pathlib.Path:
    here = pathlib.Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / "nbtools" / "config.py").is_file():
            return p
        if (p / "notebook" / "nbtools" / "config.py").is_file():
            return p / "notebook"
    raise SystemExit(
        "Không thấy `nbtools/` ở quanh đây, nên không xác định được gốc dự án.\n"
        f"Thư mục làm việc hiện tại: {here}\n"
        "Hãy mở notebook này từ bên trong folder `notebook/` đã giải nén, và mở "
        "cả folder chứ không chỉ riêng file .ipynb."
    )


ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import nbtools
from nbtools import Step, run, plan, describe, dir_summary

report = nbtools.check_environment()
data = nbtools.check_data()

#### Hai công tắc của cả notebook

Chúng nằm ở đây, và chỉ ở đây.

* **`SKIP_EXISTING`** — bước nào đã có đủ file output thì bỏ qua và in ra bảng
  output đang có. Trên máy trắng không file nào có sẵn, nên **toàn bộ pipeline
  vẫn chạy thật từ raw**; từ lần thứ hai trở đi notebook chạy lại trong vài phút.
  Muốn dựng lại đúng một bước: `run(step, force=True)`.
* **`OFFLINE`** — chặn mọi bước cần mạng. Mặc định **bật khi đã có đủ `raw/`**,
  vì khi đó Phần 1 không còn việc gì để làm và bật lên là chắc chắn không bước
  nào lén gọi API. Bước nào được đánh dấu *không bắt buộc* sẽ được bỏ qua êm;
  bước nào bắt buộc mà bị chặn sẽ dừng với lý do rõ ràng.

In [ ]:
nbtools.config.SKIP_EXISTING = True
nbtools.config.OFFLINE = data["has_raw"]

nbtools.kv({
    "SKIP_EXISTING": str(nbtools.config.SKIP_EXISTING),
    "OFFLINE": str(nbtools.config.OFFLINE),
}, headers=("Công tắc", "Đang là"))
nbtools.note("Sửa thẳng hai dòng trên rồi chạy lại cell này nếu muốn khác đi.")

### Bước 2. `selection/` — những input làm bằng tay

Bốn bảng dưới đây **không script nào sinh ra được**, và chúng quyết định phạm vi
của cả panel. Chúng đi kèm trong folder này:

| File | Nó quyết định |
|---|---|
| `importers_vn.csv` | 147 nước nhập khẩu vào panel, và lý do từng nước vào/ra |
| `eu_tariff_mapping.csv` | Mỗi nước EU đọc biểu thuế TRAINS nào trong mỗi năm (`EUN` hay chính nó) |
| `hs6_unobserved_years.csv` | Những năm mà chi tiết HS6 của một nước quá thiếu để coi là "đã quan sát" (bước 15 sinh ra) |
| `mass_death_allowlist.csv` | 16 trường hợp một nước mất >60% quan hệ trong một năm nhưng đã kiểm chứng là thật |

`mass_death_allowlist.csv` là cái đáng chú ý: `build_spells.py` **dừng hẳn** nếu
gặp một vụ chết hàng loạt không nằm trong danh sách này. Đó là chốt chặn chống
lặp lại lỗi v1 (xem T7 trong fix plan).

In [ ]:
dir_summary("selection", "*.csv")
describe("selection/importers_vn.csv", n=3)
describe("selection/mass_death_allowlist.csv", n=5)

# Dựng lại selection/ từ Comtrade: KHÔNG chạy trong lần đọc thường, vì nó ghi đè
# các bảng đã chỉnh tay. Chỉ bật khi thực sự muốn mở rộng phạm vi nghiên cứu.
REGENERATE_SELECTION = False
if REGENERATE_SELECTION:
    run(Step("02a_select_countries", "Dump khả dụng Comtrade + TRAINS",
             ["scripts/select_countries.py"],
             outputs=["selection/comtrade_da.csv", "selection/trains_avail.csv"],
             network=True, needs_key=True), force=True)
    run(Step("02b_select_importers", "Chọn nước nhập khẩu theo 3 tiêu chí",
             ["scripts/select_importers_vn.py"],
             outputs=["selection/importers_vn.csv"],
             network=True, needs_key=True), force=True)

### Bước 3. Bảng tra cứu mã HS

Hai script nhỏ nhưng là điều kiện cần của mọi thứ phía sau.

**`fetch_concordance_unsd.py`** tải bảng *correlation* của UNSD. Khác biệt với
bảng *conversion* của WITS đang có sẵn trong `data/raw/concordance/`: bảng WITS
là nhiều-về-một, mỗi mã đời mới trỏ về đúng một mã HS1992. Khi một đời HS gộp
nhiều mã cũ thành một mã mới, bảng WITS giữ lại một mã và **những mã còn lại mất
hết hậu duệ** — quan hệ nằm trong đó "chết" ngay tại năm đổi đời HS. Đó là lỗi
T1, nguồn của khoảng 1.671 ca chết giả trong v1. Bảng UNSD giữ đủ quan hệ n:n
nên bước 14 mới gộp lại được.

**`extend_eu_tariff_mapping.py`** kéo `selection/eu_tariff_mapping.csv` tới năm
cuối của panel. Bảng này dừng ở 2023 trong khi panel chạy tới 2025, và mỗi
module xử lý khoảng trống một kiểu — CBAM và EVFTA kẹp năm về 2023, còn module
thuế thì không, nên DEU/FRA/NLD năm 2024 đọc nhầm file của chính mình thay vì
`EUN` (lỗi F4).

> Bước 3b **không khai `outputs`** nên nó chạy mỗi lần. Cố ý: nó sửa một file
> tại chỗ chứ không sinh ra file mới, nên "output đã có" không nói lên điều gì.
> Chạy lại vô hại — lần thứ hai nó thêm 0 dòng.

In [ ]:
run(Step("03a_concordance", "Bảng correlation HS của UNSD",
         ["scripts/fetch_concordance_unsd.py"],
         outputs=["data/raw/concordance/unsd"],
         minutes=2, network=True,
         note="Tải các workbook correlation H1..H6 của UNSD; nhỏ, vài MB."))

run(Step("03b_eu_mapping", "Kéo eu_tariff_mapping.csv tới năm cuối panel",
         ["scripts/extend_eu_tariff_mapping.py"],
         requires=["selection/eu_tariff_mapping.csv"],
         minutes=0.1,
         note="Sửa một file trong selection/ tại chỗ; idempotent."))

describe("selection/eu_tariff_mapping.csv", n=5)

---
## Phần 1 — Thu thập raw (bước 4–13)

Mười nguồn độc lập. Mỗi nguồn một cell, vì chúng hỏng theo những cách khác nhau:
Comtrade hết quota, TRAINS trả 404, UNCTAD chặn Cloudflare, EU đổi URL của công
báo. Gộp chung một cell thì một nguồn hỏng là mất cả lượt.

**Mọi script ở phần này đều resume được.** File nào đã nằm trên đĩa thì bỏ qua,
nên cứ chạy lại cho tới khi hết lỗi.

> **Đã có `data/raw/` thì cả phần này tự bỏ qua** — mỗi cell in ra bảng file
> đang có rồi đi tiếp. Phần chữ vẫn đáng đọc: nó cho biết từng thư mục con
> trong `raw/` từ đâu mà có, và mỗi nguồn có giới hạn gì.

```
data/raw/   765 MB sau khi xong  (nguồn 10,5 GB bên UNCTAD được lọc ngay khi stream)
```

| Bước | Nguồn | Ra | Nặng |
|---|---|---|---|
| 4 | UN Comtrade — VN xuất khẩu, HS6 | `data/raw/trade/` | 3.305 file, nhiều ngày vì quota |
| 5 | Comtrade — tổng nhập khẩu thế giới + mirror | `data/raw/trade_world/`, `trade_mirror/` | 208 MB, nặng nhất |
| 6 | WITS TRAINS — thuế MFN và ưu đãi | `data/raw/tariffs/` | 61 MB |
| 7 | UNCTAD TRAINS — NTM researcher file | `data/raw/ntm/researcher/` | **58,4 phút, 1 request** |
| 8 | EU — Combined Nomenclature + GSP | `data/raw/eu_cn/`, `tariffs/pref/` | 30 MB, PDF |
| 9 | EVFTA Annex 2-A | `data/raw/evfta/` | 21 MB, PDF |
| 10 | USITC — thuế đối ứng Mỹ 2025 | `data/raw/us_tariffs_2025/` | 14 MB |
| 11 | CEPII Gravity, WDI, DESTA, TTBD, Atlas, Pink Sheet | `data/raw/{gravity,wdi,rta,ttbd,complexity,shocks}/` | 198 MB |
| 12 | Yale EPI | `data/raw/epi/` | 3 MB |
| 13 | NTM công khai của WITS (+ vĩ mô Development) | `data/raw/ntm/*.csv` → `interim/ntm_country.csv` | nhỏ, nhưng **bắt buộc** |

### Bước 4. Xuất khẩu của Việt Nam, HS6 × nước nhập khẩu × năm

Đây là xương sống của panel: mọi thứ khác chỉ là cột gắn thêm vào nó.

Đơn vị lấy về là **nước nhập khẩu tự khai** (reporter = nước nhập, partner =
Việt Nam), chứ không phải Việt Nam khai xuất. Lý do: biểu thuế và NTM đều do
nước nhập áp, nên đọc từ phía nước nhập thì thuế, NTM và kim ngạch cùng một hệ
quy chiếu.

⚠️ **Quota.** Comtrade giới hạn theo ngày. Script dừng sạch khi hết quota, giữ
nguyên file đã xong, và chạy lại là tiếp tục. Kịch bản `scripts/pull_2022_2024.sh`
là ví dụ cách chạy ba lượt liên tiếp có xử lý quota.

⚠️ **12 file bị đếm đôi** (BRA 2002–2007, CHN 2015–2017, NIC 2016–2017, BDI 2010)
đã được tải lại trong v2 — chúng chứa cả bảng chia theo thủ tục hải quan *và*
dòng tổng, nên mọi giá trị gấp đúng 2 lần (lỗi T5).

In [ ]:
run(Step("04_trade_vn", "Comtrade: VN xuất khẩu, HS6 × nước nhập × năm",
         ["scripts/fetch_trade.py", "--pass", "vn"],
         outputs=["data/raw/trade"],
         network=True, needs_key=True,
         note="Hết quota thì thoát mã 1 và giữ nguyên file đã tải — chạy lại cell này hôm sau."))

dir_summary("data/raw/trade", "*.csv.gz")

### Bước 5. Mẫu số: tổng nhập khẩu của thế giới, và bản mirror

Hai lượt tải nữa từ cùng một API nhưng trả lời hai câu khác:

* **`--pass world`** — mỗi nước nhập khẩu mặt hàng đó *từ toàn thế giới* bao
  nhiêu. Đây là mẫu số của `vn_market_share_pct`, `log_total_import_cp` và
  chỉ số RCA. Nặng nhất trong ba lượt và hay đụng quota nhất.
* **`--pass mirror`** — Việt Nam tự khai xuất khẩu. Không vào panel; nó là phép
  đối chứng: khi một nước ngừng khai báo, mirror cho biết dòng hàng có thật sự
  dừng hay chỉ là nước đó thôi nộp số liệu. Bước 16 dùng nó để phân biệt
  "chết thật" với "mất quan sát".

In [ ]:
run(Step("05a_trade_world", "Comtrade: tổng nhập khẩu từ thế giới (mẫu số)",
         ["scripts/fetch_trade.py", "--pass", "world"],
         outputs=["data/raw/trade_world"],
         network=True, needs_key=True,
         note="Lượt nặng nhất — 208 MB, 1.880 importer-year."))

run(Step("05b_trade_mirror", "Comtrade: VN tự khai xuất khẩu (đối chứng)",
         ["scripts/fetch_trade.py", "--pass", "mirror"],
         outputs=["data/raw/trade_mirror"],
         network=True, needs_key=True))

dir_summary("data/raw/trade_world", "*.csv.gz")

### Bước 6. Thuế: MFN và ưu đãi, từ WITS TRAINS

Một quan hệ được gắn **thuế mà nước nhập áp lên mặt hàng đó**, nên reporter
TRAINS chính là nước nhập khẩu — trừ các nước EU, vốn đọc biểu `EUN` chung theo
`selection/eu_tariff_mapping.csv`.

Hai giới hạn đã biết, ghi ra đây để đừng ai tưởng là lỗi:

* **TRAINS dừng ở 2023.** Nó trả 404 cho mọi reporter năm 2024 và 2025
  (kiểm lại 29/08/2026). Khoảng trống này được vá ở bước 8 + 20 bằng chính công
  báo CN của EU.
* **Ưu đãi theo mã nhóm chưa lấy.** Nhiều biểu ưu đãi nộp dưới mã nhóm TRAINS
  (`N52`, `C02`, `G27`…) chứ không dưới mã Việt Nam. Hệ quả: `tariff_rate` của
  MYS, SGP, THA, IDN, PHL, KHM, BRN, IND, NZL, CAN, MEX, PER, HKG **100% là
  MFN** (lỗi F2, hoãn theo quyết định D3 — B0 chỉ cần EU). Đừng đọc cột đó như
  thuế thực trả ở các nước này.

In [ ]:
run(Step("06_tariffs", "WITS TRAINS: biểu thuế MFN + ưu đãi",
         ["scripts/fetch_tariffs.py", "--pass", "all"],
         outputs=["data/raw/tariffs/mfn", "data/raw/tariffs/pref"],
         requires=["selection/eu_tariff_mapping.csv"],
         network=True,
         note="8.078 cặp (reporter, năm) sau khi gộp EU. Không cần API key."))

dir_summary("data/raw/tariffs/mfn", "*.gz")

### Bước 7. NTM ở cấp HS6 — file researcher của UNCTAD

Câu chuyện đáng kể nhất của cả pipeline, vì nó lật một kết luận sai.

Kết luận cũ của dự án là không thể lấy dữ liệu này từ máy này. Đúng với **cánh
cửa** đã thử, sai với **dữ liệu**: route `POST /denormalisedMeasures` chặn
`pageSize` ở 20 và nằm sau một rule Cloudflare đã trả 429 sáu lần trong 31 phút
backoff — vài nghìn trang thì không bao giờ xong.

Cùng cơ sở dữ liệu đó được công bố nguyên khối qua một GET không giới hạn tốc độ:
`https://api-trains2.unctad.org/get-researcher-file/2`, **10.548.645.866 byte,
không cần đăng nhập**. `fetch_ntm_researcher.py` stream và lọc ngay trong lúc tải,
vì host không hỗ trợ gzip lẫn Range — không resume được, và cũng không có lý do
gì để chứa 10,5 GB trên đĩa.

Kết quả một lượt: **58,4 phút, một request, không bị chặn**. Đọc 153.058.539
dòng, giữ 18.686.715 (12,2%) sau khi lọc về `WLD` + Việt Nam + nhóm đối thủ
cạnh tranh, còn **66,8 MB** nén gzip.

⚠️ Chạy lại là tải lại từ đầu 10,5 GB. Kiểm `data/raw/ntm/researcher/` trước.

In [ ]:
run(Step("07_ntm_researcher", "UNCTAD TRAINS: NTM researcher file (10,5 GB stream)",
         ["scripts/fetch_ntm_researcher.py"],
         outputs=["data/raw/ntm/researcher"],
         minutes=58.4, network=True,
         note="Không resume được. Đã đo: 58,4 phút cho 10.548.645.866 byte."))

dir_summary("data/raw/ntm/researcher")

### Bước 8. EU: công báo Combined Nomenclature + biểu GSP

Hai mảnh vá cho lỗ hổng của TRAINS ở phía EU:

* **`fetch_eu_cn.py`** tải công báo CN 2024–2026. Annex I của nó là Biểu thuế
  quan chung, và cột 3 của bảng thuế chính là mức MFN. Đây là cách duy nhất để
  hai năm cuối panel có thuế EU *của riêng năm đó* thay vì kéo mức 2023 sang.
* **`fetch_eu_gsp.py`** lấy biểu GSP của EU dưới **mã nhóm**. Trước EVFTA, hàng
  Việt Nam không vào EU theo MFN — Việt Nam là nước hưởng GSP tiêu chuẩn. TRAINS
  có nộp biểu này, nhưng nộp dưới mã nhóm chứ không dưới mã Việt Nam, nên lượt
  tải ở bước 6 không thấy.

> Bước 8b ghi chung thư mục với bước 6 (`data/raw/tariffs/pref/`), nên nếu khai
> output là cả thư mục thì nó sẽ luôn bị coi là "đã xong" ngay sau bước 6. Ở
> đây output trỏ vào đúng một file GSP để việc bỏ qua nói đúng sự thật.

In [ ]:
run(Step("08a_eu_cn", "Công báo Combined Nomenclature của EU, 2024-2026",
         ["scripts/fetch_eu_cn.py"],
         outputs=["data/raw/eu_cn"],
         minutes=5, network=True))

run(Step("08b_eu_gsp", "Biểu GSP của EU trên TRAINS (mã nhóm)",
         ["scripts/fetch_eu_gsp.py"],
         outputs=["data/raw/tariffs/pref/EUN_2000_G26.csv.gz"],
         minutes=5, network=True,
         note="Ghi chung vào data/raw/tariffs/pref/ với bước 6."))

dir_summary("data/raw/eu_cn")

### Bước 9. EVFTA Annex 2-A

Annex 2-A là phụ lục xoá bỏ thuế quan của Hiệp định EU–Việt Nam. **Appendix
2-A-1 là biểu của phía EU** — tức mức thuế mà nhà xuất khẩu Việt Nam thật sự đối
mặt — và là cái mà chiến lược nhận dạng của nghiên cứu cần. Các phụ lục còn lại
vẫn tải về để đối chiếu.

Không ai công bố phụ lục này dưới dạng dữ liệu. Nó là PDF, và bước 19 là nơi
biến nó thành bảng. **Biến điều trị của cả thiết kế nghiên cứu sinh ra từ đây.**

In [ ]:
run(Step("09_evfta_annex", "Tải EVFTA Annex 2-A và 5 phụ lục",
         ["scripts/fetch_evfta_annex.py"],
         outputs=["data/raw/evfta"],
         minutes=3, network=True))

dir_summary("data/raw/evfta")

### Bước 10. Thuế đối ứng của Mỹ năm 2025

Brief dữ liệu đặt hành động 2025 của Mỹ vào trung tâm thiết kế — "46% → 10% →
20%/40%" — và TRAINS không có gì cả: WITS trả 404 cho mọi năm thuế 2024 và 2025.

`fetch_us_tariffs_2025.py` lấy thẳng từ USITC: 109 dòng thuế chương 99 mang mức
thuế theo nước, trên 85 nước. Nhưng payload REST **chỉ có mức, không có phạm vi
sản phẩm miễn trừ** — danh sách miễn trừ nằm trong một ghi chú pháp lý (U.S. note
2(v)(iii)) chỉ công bố dạng PDF. Bước 21 mở nốt phần đó, và bước 23c biến mức
thuế thành bốn cột mô tả năm 2025.

In [ ]:
run(Step("10_us_tariffs", "USITC: thuế đối ứng Mỹ 2025, theo tháng",
         ["scripts/fetch_us_tariffs_2025.py"],
         outputs=["data/raw/us_tariffs_2025"],
         minutes=5, network=True))

dir_summary("data/raw/us_tariffs_2025")

### Bước 11. Biến kiểm soát: gravity, WDI, DESTA, TTBD, Atlas, Pink Sheet

Sáu nguồn công khai, không nguồn nào cần tài khoản — đó chính là lý do chúng
được chọn, khác với TRAINS Online (đăng nhập Azure AD) hay WTO I-TIP.

| Nguồn | Cho panel cái gì |
|---|---|
| CEPII Gravity V202211 | khoảng cách, chung biên giới, chung ngôn ngữ, thuộc địa cũ |
| World Bank WDI | GDP, dân số, tỷ giá, lạm phát, CO₂, LPI |
| DESTA | hiệp định thương mại có hiệu lực với Việt Nam từ năm nào |
| Temporary Trade Barriers Database | vụ kiện chống bán phá giá / trợ cấp / tự vệ |
| Harvard Growth Lab Atlas | độ phức tạp sản phẩm (PCI) và quốc gia (ECI) |
| World Bank Pink Sheet + EPU | giá hàng hoá và chỉ số bất định chính sách |

⚠️ **CEPII Gravity dừng ở 2021**, panel chạy tới 2025 — bốn năm kiểm soát treo
vào đó. `gravity_source_year` đánh dấu dòng nào là giá trị kéo sang.
⚠️ **TTBD dừng ở 2015Q4.** Quyết định D4: từ 2016 để **null**, không để 0, và
giữ cờ `ttbd_observed`. Đọc 0 ở đây là đọc sai.

In [ ]:
run(Step("11_covariates_raw", "CEPII + WDI + DESTA + TTBD + Atlas + Pink Sheet",
         ["scripts/fetch_covariates.py"],
         outputs=["data/raw/gravity", "data/raw/wdi", "data/raw/rta",
                  "data/raw/ttbd", "data/raw/complexity", "data/raw/shocks"],
         minutes=25, network=True,
         note="Gravity là file nặng nhất (198 MB)."))

for d in ["data/raw/gravity", "data/raw/wdi", "data/raw/rta",
          "data/raw/ttbd", "data/raw/complexity", "data/raw/shocks"]:
    dir_summary(d)

### Bước 12. EPI của Yale, dạng chuỗi theo năm

`build_glpi.py` ở bước 23b đọc `data/raw/epi/epi2026results.xlsx` — **một lát
cắt ngang duy nhất**, và đó chính là lý do `importer_glpi_*` không nhúc nhích
theo thời gian. Script này tải về **nguyên liệu** để sửa: kho lưu trữ của Yale
không công bố điểm EPI của các năm cũ (không ai công bố), nhưng có công bố từng
chỉ số thành phần dưới dạng chuỗi năm từ 1996.

Nó dừng lại đúng ở chỗ đó, có chủ ý: gộp ~50 chỉ số thành một điểm EPI theo năm
là **một quyết định nghiên cứu**, và dự án đã có sẵn một câu hỏi chưa chốt cùng
loại (chọn biến thể GLPI nào ở bước 23b). Chồng thêm một chỉ số tổng hợp tự chế
nữa là giấu một quyết định nghiên cứu vào trong một bước dữ liệu.

**Nên: `epi_indicators_annual.csv` hiện chưa có module nào đọc.** Nó nằm đó chờ
người quyết định cách gộp. `importer_glpi_*` trong panel vẫn là bất biến theo
thời gian — đây là hạn chế số 6 ở bước 32, không phải lỗi.

> Vì không module nào đọc output của nó, bước này được đánh dấu **không bắt
> buộc**: bị chặn thì notebook cảnh báo rồi đi tiếp. `network=True` cũng chỉ bật
> khi chưa có file trên đĩa — script tự bỏ qua lượt tải nếu `data/raw/epi/*.zip`
> đã có, nên với bản gửi kèm nó chạy được ngoại tuyến.
>
> (Lưu ý `data/raw/epi/epi2026results.xlsx` thì **có** cần: bước 23b đọc nó. Nó
> nằm trong bản gửi kèm, và `requires` của 23b sẽ bắt nếu thiếu.)

In [ ]:
have_epi_zip = nbtools.exists("data/raw/epi/epi2026indicatorsna.zip")

run(Step("12_epi", "Yale EPI dạng chuỗi năm",
         ["scripts/fetch_epi_annual.py"],
         outputs=["data/raw/epi", "data/interim/epi_indicators_annual.csv"],
         minutes=0.5, network=not have_epi_zip, optional=True,
         note="Đã có sẵn zip trên đĩa nên không cần mạng." if have_epi_zip else ""))

describe("data/interim/epi_indicators_annual.csv", n=3)

### Bước 13. `ntm_country.csv` + vĩ mô WITS Development

`fetch_macro.py` làm hai việc độc lập nhau, và chúng **không** cùng mức quan trọng.

**Nửa NTM — bắt buộc.** Nó lấy ba file NTM công khai của WITS và gấp file
*coverage* thành `data/interim/ntm_country.csv`. File nhỏ này là thứ mà
`build_ntm.py` (bước 22b) **và** `merge_panel.py` (bước 24) đều bắt buộc phải
có: thiếu nó, cả hai thoát với `ntm_country.csv missing - run fetch_macro.py
first`. Không script nào khác sinh ra nó.

Ba file NTM đó **không** đủ cho thiết kế: chúng là một lát cắt ngang ở cấp
ngành, mọi sản phẩm trong một nhóm 16 ngành mang cùng một giá trị ở mọi năm, và
20% quan hệ không nhận được gì — Trung Quốc và Hàn Quốc nằm trong số đó. Bước 22a
thay chúng bằng file researcher của bước 7. Chúng vẫn được dựng để so sánh và để
các cột `ntm_sector_*` có mặt.

**Nửa vĩ mô — không bắt buộc.** `macro_panel.csv` lấy từ WITS Trade Stats –
Development chỉ là *phương án dự phòng* trong `merge_panel.py`: merge ưu tiên
`macro_panel_v2.csv`, do bước 23a dựng thẳng từ `data/raw/wdi/` (World Bank API,
phủ nhiều chỉ số hơn và tới năm mới hơn).

> **Cờ `--ntm-only` là thứ làm cho bản chỉ có `raw/` chạy được.** Nửa vĩ mô phải
> gọi API; nửa NTM thì không — ba file NTM đã nằm sẵn trong `data/raw/ntm/`, và
> `fetch_macro.py` đọc lại chúng từ đĩa thay vì tải lại (giống
> `fetch_epi_annual.py` và `fetch_cbam_scope.py`). Cell dưới tự chọn: ngoại
> tuyến thì `--ntm-only`, có mạng thì chạy cả hai nửa.

In [ ]:
have_ntm_raw = nbtools.exists("data/raw/ntm/NTM-Trade-Frequency-Coverage-Ratio.csv")
ntm_only = nbtools.config.OFFLINE and have_ntm_raw

run(Step("13_macro_ntm", "ntm_country.csv (+ vĩ mô WITS nếu có mạng)",
         ["scripts/fetch_macro.py"] + (["--ntm-only"] if ntm_only else []),
         outputs=(["data/interim/ntm_country.csv"] if ntm_only else
                  ["data/interim/ntm_country.csv", "data/interim/macro_panel.csv"]),
         requires=["data/raw/ntm/NTM-Trade-Frequency-Coverage-Ratio.csv"]
                  if ntm_only else [],
         minutes=0.5 if ntm_only else 20, network=not ntm_only,
         note="Ngoại tuyến: chỉ dựng lại ntm_country.csv từ data/raw/ntm/ đã có."
              if ntm_only else
              "Chạy cả hai nửa; nửa vĩ mô gọi WITS Development nên cần mạng."))

describe("data/interim/ntm_country.csv", n=3)

if nbtools.exists("data/interim/macro_panel.csv"):
    describe("data/interim/macro_panel.csv", n=3)
else:
    nbtools.note("Không có `macro_panel.csv`, và đó là chuyện bình thường ở chế độ "
                 "ngoại tuyến — bước 23a dựng `macro_panel_v2.csv` thay cho nó.")

---
## Phần 2 — Tiền xử lý: từ raw thành các bảng trung gian (bước 14–23)

Phần này không gọi API nữa. Nó biến `data/raw/` thành `data/interim/`: trước hết
là **khoá sản phẩm** và **spell** (bước 14–18), sau đó từng module đặc trưng gắn
vào spell (bước 19–23).

Thứ tự bắt buộc: `families` → `spells` → mọi thứ còn lại. Lý do là mọi module
đều khoá theo `product_family`, và `build_ntm6.py` còn đọc thẳng `spells.csv`.
Từ đây trở đi mỗi bước khai cả `requires` — input nó phải đọc — nên nếu bạn chạy
lệch thứ tự, notebook dừng ngay với tên file còn thiếu thay vì để script chết
giữa chừng.

```
data/raw/ 765 MB  ─────►  data/interim/ 1,2 GB
```

### Bước 14. `families.py` — một định nghĩa duy nhất cho khoá sản phẩm

**Family** là đơn vị sản phẩm của panel: một mã HS1992 (H0), hoặc một nhóm nhỏ
các mã H0, cùng với mọi mã đời sau thuộc về nó.

Tại sao phải có khái niệm này thay vì dùng thẳng HS6: cứ mỗi lần HS đổi đời
(2002, 2007, 2012, 2017, 2022), một số mã bị gộp. Trong bảng conversion của
WITS, chỉ một mã giữ được hậu duệ, các mã còn lại mất sạch — và **1.982 trên
1.983 dòng bị phơi ra đã "chết" đúng tại năm đổi đời**, ở mọi nước, mọi lần đổi.
Đó không phải quan hệ tan vỡ, đó là cái thước bị đổi.

Cách xử lý (quyết định D1, phương án lai):
1. gộp mã mồ côi với mã nhận nó, **nhưng chỉ trong cùng một HS4**;
2. mã nào vẫn không gộp được thì **kiểm duyệt (censor) tại năm đổi đời**, chứ
   không tính là chết.

Ràng buộc "cùng HS4" là thứ làm phương án này dùng được: không ràng buộc thì
thành phần liên thông thuần tuý tạo ra một family gồm 1.368 mã trải khắp hàng
chục chương. Có ràng buộc, family lớn nhất còn **10 mã**, và **70% dòng bị phơi
ra được giải quyết** (EU: 73%).

Trước đây khoá này có **ba bản cài đặt khác nhau** trong ba script
(`fetch_cbam_scope`, `extract_us_exemptions`, `build_ntm6`). Chúng đã được gom về
đây (lỗi K1) — nếu không, mọi thay đổi về family sẽ âm thầm làm lệch khoá.

In [ ]:
run(Step("14_families", "Dựng khoá sản phẩm (family) từ bảng correlation",
         ["scripts/families.py"],
         outputs=["data/interim/family_map.csv",
                  "data/interim/family_members.csv",
                  "data/interim/family_merges.csv"],
         requires=["data/raw/concordance/unsd", "data/raw/concordance/H6_to_H0"],
         minutes=0.5,
         note="FAMILIES_WITS_ONLY=1 trong env sẽ tái lập khoá kiểu v1 (để đối chứng)."))

describe("data/interim/family_map.csv", n=5)

# Có bao nhiêu family là gộp từ nhiều mã H0, và mã mồ côi nào phải censor?
import polars as pl

nbtools.md("**Mã mồ côi ở các lần đổi đời HS — gộp được hay phải kiểm duyệt:**")
nbtools.frame(nbtools.value_counts("data/interim/family_merges.csv", "action"))

nbtools.md("**Family gồm nhiều mã H0 (`merged`) so với family một mã:**")
nbtools.frame(pl.read_csv(nbtools.abs_path("data/interim/family_members.csv"))
                .group_by("merged").len().sort("merged"))

### Bước 15. Năm nào của nước nào thì chi tiết HS6 quá thiếu để tin

Một nước có thể khai đủ kim ngạch ở mức TOTAL mà bỏ trống phần lớn ở mức HS6 —
dòng hàng bảo mật, hoặc một năm khai chủ yếu dưới mã không phân bổ. Năm đó
**không phải** là "Việt Nam bán được ít": chi tiết sản phẩm đơn giản là không có,
và đọc nó như đã quan sát thì giết sạch mọi quan hệ rơi ra ngoài vài dòng được
khai.

ALB 2014 là ca rõ nhất: **7 dòng HS6, phủ 47% tổng của chính nó**.

Script chấm điểm mọi importer-year theo tỷ lệ HS6/TOTAL và ghi ra
`selection/hs6_unobserved_years.csv`. 13 importer-year có độ phủ dưới 50%.
Trung Quốc từ 2015 (0,73–0,94) là độ phủ thiếu *dai dẳng* chứ không phải sự cố,
nên vẫn tính là đã quan sát.

> File này đi kèm sẵn trong `selection/`, nên bước này thường được bỏ qua. Nó
> vẫn ở đây vì nếu bạn mở rộng `data/raw/trade/` thì phải chấm lại: gọi
> `run(step, force=True)`.

In [ ]:
run(Step("15_hs6_coverage", "Chấm độ phủ HS6 so với tổng của chính nước đó",
         ["scripts/screen_hs6_coverage.py"],
         outputs=["selection/hs6_unobserved_years.csv"],
         requires=["data/raw/trade"],
         minutes=3))

describe("selection/hs6_unobserved_years.csv", n=10)

### Bước 16. `build_spells.py` — biến panel HS6 thành dữ liệu sinh tồn

Bước nặng nhất về RAM của cả pipeline: **đo được 3 phút 09, đỉnh 2,8 GB**.

Một **spell** là đời sống của một quan hệ (nước nhập khẩu j × family k) cho hàng
Việt Nam. Spell bắt đầu năm đầu tiên Việt Nam xuất family đó sang nước đó, và kết
thúc năm dòng hàng dừng lại — **với điều kiện những năm xác nhận việc dừng đó
thật sự đã được công bố**.

Mệnh đề điều kiện ấy là toàn bộ khác biệt giữa v1 và v2. Trong v1 nó không có, và
panel sinh ra 18.790 ca chết giả. Quy tắc v2, chung cho các lỗi T2–T4:

> Một cái chết ở năm E chỉ là sự kiện nếu các năm E+1 … E+1+GAP **đều đã được
> công bố ở mức HS6**. Không đủ thì spell bị kiểm duyệt, và `censor_reason` ghi
> lý do. Điểm bắt đầu cũng soi gương như vậy qua `start_reason` và `left_trunc`.

`GAP_TOLERANCE = 1`: cần hai năm vắng mặt mới tính là chết. Hệ quả là ở năm áp
chót không thể xác nhận cái chết nào — nên 2025 có 0 sự kiện, đúng theo thiết kế
chứ không phải thiếu dữ liệu (quyết định D2).

**Chốt chặn chết hàng loạt (T7):** script *dừng hẳn* khi hơn 60% quan hệ đang
sống của một nước (n ≥ 30) chết trong cùng một năm, trừ khi
`selection/mass_death_allowlist.csv` ghi sẵn lý do đã kiểm chứng. Hiện còn 16 ca,
mỗi ca đã đối chiếu với TOTAL của chính nước đó và với bản mirror của Việt Nam.

In [ ]:
run(Step("16_spells", "Dựng spell + episode từ panel HS6",
         ["scripts/build_spells.py"],
         outputs=["data/interim/spells.csv",
                  "data/interim/episodes.csv",
                  "data/interim/importer_year_revision.csv"],
         requires=["data/interim/family_map.csv",
                   "data/raw/trade", "data/raw/trade_world", "data/raw/trade_mirror",
                   "selection/hs6_unobserved_years.csv",
                   "selection/mass_death_allowlist.csv"],
         minutes=2,
         note="Bản v2 đo 3'09 với đỉnh RAM 2,8 GB; lượt dựng lại 22/09/2026 mất 1'54. "
              "Dừng có kiểm soát nếu gặp chết hàng loạt."))

### Bước 17. Đọc `spells.csv` — quy tắc kiểm duyệt có thật sự cắn không

Cell này không chạy gì; nó đọc kết quả bước 16 và trả lời: **mỗi spell kết thúc
vì lý do gì?** Đây là chỗ nhìn thấy tận mắt sự khác biệt v1 → v2.

Đối chiếu với con số đã chốt của bản v2:

| `censor_reason` | Spell | Nghĩa |
|---|---:|---|
| *(null)* | 112.885 | chết thật, đã xác nhận — đây là `event = 1` |
| `window_end` | 63.920 | vẫn sống khi panel hết (2025) |
| `window_edge` | 9.500 | chết quá sát mép, không đủ năm để xác nhận |
| `unobserved_year` | 2.983 | năm xác nhận không được công bố |
| `hs_revision` | 542 | mã mồ côi ở lần đổi đời HS, không gộp được |

In [ ]:
nbtools.describe("data/interim/spells.csv", n=3, show_columns=20)
nbtools.unique_key("data/interim/spells.csv", ["spell_id"])

nbtools.md("**Lý do kết thúc spell (`censor_reason`):**")
nbtools.frame(nbtools.value_counts("data/interim/spells.csv", "censor_reason", 10))

nbtools.md("**Lý do điểm bắt đầu không chắc (`start_reason`):**")
nbtools.frame(nbtools.value_counts("data/interim/spells.csv", "start_reason", 10))

### Bước 18. Đọc `episodes.csv` — bung spell ra thành năm quan sát

`episodes.csv` là dạng đếm-năm của spell: mỗi dòng một năm của một quan hệ, với
`t_start`/`t_stop` và `event`. Đây là hình dạng mà mô hình sinh tồn ăn vào, và là
bộ khung mà toàn bộ Phần 3 sẽ gắn cột lên.

`build_spells.py` cũng tính luôn nhóm biến phái sinh ở mức HS6 ngay tại đây, vì
chúng cần cả bảng thế giới của bước 5: thị phần của Việt Nam, RCA, tăng trưởng
của chính mặt hàng, HHI thị trường và HHI sản phẩm.

In [ ]:
nbtools.describe("data/interim/episodes.csv", n=3, show_columns=24)
nbtools.unique_key("data/interim/episodes.csv", ["spell_id", "year"])

nbtools.md("**Biến phái sinh tính ngay ở bước này (mức HS6, cần bảng thế giới):**")
nbtools.frame(nbtools.null_share(
    "data/interim/episodes.csv",
    ["rca", "vn_market_share_pct", "total_import_cp_usd",
     "country_growth_pct", "world_growth_pct", "hhi_market", "hhi_product"]))

### Bước 19. EVFTA: từ PDF thành lịch trình cắt thuế

`build_evfta_staging.py` bóc Appendix 2-A-1 — biểu của phía EU — thành dữ liệu:
mỗi dòng CN8 kèm **mức thuế cơ sở** (Biểu thuế quan chung có hiệu lực ngày
26/06/2012) và **nhóm lộ trình** (staging category: A, B3, B5, B7, B10…).

Bốn output, đi từ chi tiết nhất tới cái panel thật sự dùng:

| File | Grain |
|---|---|
| `evfta_eu_schedule_cn8.csv` | CN8 — như trong phụ lục |
| `evfta_staging_hs6.csv` | HS6 |
| `evfta_tariff_path_hs6.csv` | HS6 × năm — đường thuế thực tế theo lộ trình |
| `evfta_staging_family.csv` | family — khoá của panel |

Cột `staging_cat` là **biến điều trị** trong thiết kế B6: nó được đọc như phân
bổ điều trị đã định trước, nên không có bản `_lag1`.

In [ ]:
run(Step("19_evfta", "Bóc EVFTA Annex 2-A thành lịch trình cắt thuế",
         ["scripts/build_evfta_staging.py"],
         outputs=["data/interim/evfta_eu_schedule_cn8.csv",
                  "data/interim/evfta_staging_hs6.csv",
                  "data/interim/evfta_tariff_path_hs6.csv",
                  "data/interim/evfta_staging_family.csv"],
         requires=["data/raw/evfta", "data/interim/family_map.csv"],
         minutes=1.5, note="Đọc PDF bằng pdfplumber."))

nbtools.describe("data/interim/evfta_staging_family.csv", n=5)
nbtools.frame(nbtools.value_counts("data/interim/evfta_staging_family.csv",
                                   "staging_cat", 12))

### Bước 20. Một biểu thuế EU duy nhất cho 2002–2035

Không nguồn nào phủ hết cửa sổ, nên `build_eu_tariff_panel.py` ghép bốn mảnh:

| Giai đoạn | Nguồn |
|---|---|
| MFN 2002–2023 | TRAINS (`data/raw/tariffs/mfn/EUN_*.csv.gz`) — bước 6 |
| MFN 2024–2026 | công báo CN của EU — bước 8, bóc bởi `build_eu_mfn_cn.py` |
| GSP 2000–2014 | biểu nhóm TRAINS — bước 8b |
| EVFTA 2020→ | đường thuế của bước 19 |

⚠️ **Bước 20a có hai chế độ, và cell dưới tự chọn đúng chế độ.**
`build_eu_mfn_cn.py` không cờ sẽ parse lại các PDF công báo (**~10 phút, ~6 GB
RAM**) và sinh ra `eu_mfn_cn8.csv` → `eu_mfn_hs6.csv` → `eu_mfn_family.csv`. Cờ
`--families-only` bỏ phần PDF và chỉ gấp lại từ `eu_mfn_hs6.csv` **đã có sẵn** —
nhanh hơn nhiều, nhưng nếu file đó chưa tồn tại thì script chết. Trên máy chỉ mới
có `raw/`, `eu_mfn_hs6.csv` chưa tồn tại, nên phải chạy bản đầy đủ đúng một lần.

> Nếu máy bạn dưới 8 GB RAM, chạy riêng cell này khi không mở gì khác.

Kết quả (`eu_tariff_panel.csv`) là chỗ duy nhất bước 27 đọc **độ trễ chính sách
EU**. Đọc độ trễ từ chính các dòng của panel thì 7.123 episode-year mất mức thuế
mà lịch biểu vốn có: một mức thuế tồn tại bất kể năm ngoái có nước EU nào nhập
family đó hay không.

In [ ]:
# --families-only chỉ gấp lại từ eu_mfn_hs6.csv; không có file đó thì phải parse
# PDF. Chọn theo tình trạng thật của đĩa, không theo giả định.
have_hs6 = nbtools.exists("data/interim/eu_mfn_hs6.csv")

if have_hs6:
    step_20a = Step("20a_eu_mfn_cn", "Gấp MFN của công báo CN về mức family",
                    ["scripts/build_eu_mfn_cn.py", "--families-only"],
                    outputs=["data/interim/eu_mfn_family.csv"],
                    requires=["data/interim/eu_mfn_hs6.csv",
                              "data/interim/family_map.csv"],
                    minutes=1,
                    note="eu_mfn_hs6.csv đã có nên chỉ gấp lại về family.")
else:
    step_20a = Step("20a_eu_mfn_cn", "Parse công báo CN -> MFN 2024-2026",
                    ["scripts/build_eu_mfn_cn.py"],
                    outputs=["data/interim/eu_mfn_cn8.csv",
                             "data/interim/eu_mfn_hs6.csv",
                             "data/interim/eu_mfn_family.csv"],
                    requires=["data/raw/eu_cn", "data/interim/family_map.csv"],
                    minutes=8,
                    note="Chưa có eu_mfn_hs6.csv nên phải bóc PDF công báo: "
                         "đo được 7'54 — bước dài nhất của lượt dựng lại từ raw. "
                         "Chỉ cần một lần.")

run(step_20a)

run(Step("20b_eu_tariff_panel", "Ghép 4 nguồn thành biểu thuế EU 2002-2035",
         ["scripts/build_eu_tariff_panel.py"],
         outputs=["data/interim/eu_tariff_panel.csv"],
         requires=["data/interim/eu_mfn_family.csv",
                   "data/interim/evfta_staging_family.csv",
                   "data/interim/family_map.csv",
                   "data/raw/tariffs/mfn", "data/raw/tariffs/pref"],
         minutes=0.2))

nbtools.describe("data/interim/eu_tariff_panel.csv", n=5, show_columns=15)

### Bước 21. Phạm vi sản phẩm của hai cú sốc chính sách

Hai script, cùng một dạng việc: biến một văn bản pháp lý thành danh sách family.

* **`fetch_cbam_scope.py`** — phạm vi sản phẩm của CBAM (cơ chế điều chỉnh biên
  giới carbon của EU). Brief giao cho EU một vai cụ thể: nó là nguồn của **cú
  sốc còn lại**, để so sánh nguy cơ từ một sắc thuế thông thường với nguy cơ từ
  một biện pháp carbon. Trước 25/08/2026, dự án chưa từng thu thập cái này —
  "CBAM" không xuất hiện trong bất kỳ script, tài liệu hay cột nào.
* **`extract_us_exemptions.py`** — danh sách miễn trừ của thuế đối ứng Mỹ 2025,
  bóc từ U.S. note 2(v)(iii) chương 99. Đây là phần bước 10 không lấy được.

⚠️ **CBAM trong cửa sổ panel là chi phí tuân thủ, không phải giá.** Điều 32 đặt
giai đoạn chuyển tiếp từ 01/10/2023 đến 31/12/2025 — chỉ *báo cáo*, không chứng
chỉ, không thanh toán. Chế độ chính thức bắt đầu 01/01/2026, một năm sau khi
panel kết thúc. Bảng nào so CBAM với thuế Mỹ 2025 phải nói rõ điều này.

> `fetch_cbam_scope.py` giữ lại bản văn bản trong `data/raw/cbam/` và bỏ qua lượt
> tải nếu đã có, nên với bản gửi kèm nó chạy được ngoại tuyến.

In [ ]:
have_cbam_src = nbtools.exists("data/raw/cbam/reg_2023_956.xhtml")

run(Step("21a_cbam", "Phạm vi sản phẩm CBAM của EU",
         ["scripts/fetch_cbam_scope.py"],
         outputs=["data/interim/cbam_products_cn.csv",
                  "data/interim/cbam_products.csv"],
         requires=["data/interim/family_map.csv"],
         minutes=0.2, network=not have_cbam_src,
         note="Văn bản đã có trong data/raw/cbam/ nên không cần mạng."
              if have_cbam_src else ""))

run(Step("21b_us_exempt", "Danh sách miễn trừ thuế đối ứng Mỹ 2025",
         ["scripts/extract_us_exemptions.py"],
         outputs=["data/interim/us_tariff_exemptions_2025_hs8.csv",
                  "data/interim/us_tariff_exemptions_2025.csv",
                  "data/interim/us_exempt_products.csv"],
         requires=["data/raw/us_tariffs_2025", "data/interim/family_map.csv"],
         minutes=0.2))

nbtools.describe("data/interim/cbam_products.csv", n=5)
nbtools.describe("data/interim/us_exempt_products.csv", n=5)

### Bước 22. NTM: ba cách đo, ba mức tin cậy khác nhau

| Script | Ra | Đơn vị | Dùng để |
|---|---|---|---|
| `build_ntm6.py` | `ntm6_observed.csv` | nước áp × family × năm khảo sát | **biến chính** |
| `build_ntm.py` | `ntm_by_type/sector/country.csv` | ngành (lát cắt ngang) | đối chứng, giữ cột cũ |
| `build_ntm_ave.py` | `ntm_ave_vn.csv` | tương đương thuế (%) | biến duy nhất so được với `tariff_rate` |

`build_ntm6.py` phủ **95,2% episode** so với 79,9% của bản cấp ngành, và lần đầu
có Trung Quốc, Hàn Quốc, Mỹ — ba nước mà file WITS công khai không phủ chút nào.

⚠️ **95,2% là độ phủ, không phải phép đo.** Chỉ **36,6%** episode nằm đúng năm mà
nước đó thật sự nộp khảo sát. Trong v1, 29,0% mang con số mượn từ một năm khảo
sát *về sau* — mọi episode 2003–2009 đều vậy, vì file bắt đầu từ 2010. Theo
quyết định D5, v2 để **null** những năm trước đợt khảo sát đầu tiên thay vì mượn
ngược (lỗi F6). Cột `ntm6_source_year` luôn cho biết con số đến từ năm nào.

`build_ntm_ave.py` cần `data/raw/ntm/ave_gtap/UNCTADGTAP11_AVEborder.csv`
(UNCTAD–GTAP 11). **Không script nào tải được file này** — nó đi kèm trong bản
gửi kèm `raw/`. Nếu thiếu, bước 22c dừng và nói rõ; xin lại file rồi chạy tiếp.

In [ ]:
run(Step("22a_ntm6", "NTM mức HS6 từ file researcher",
         ["scripts/build_ntm6.py"],
         outputs=["data/interim/ntm6_observed.csv"],
         requires=["data/raw/ntm/researcher", "data/interim/spells.csv",
                   "data/interim/family_map.csv",
                   "selection/eu_tariff_mapping.csv"],
         minutes=1,
         note="Đọc spells.csv, nên phải chạy sau bước 16."))

run(Step("22b_ntm_sector", "NTM cấp ngành từ file WITS công khai (đối chứng)",
         ["scripts/build_ntm.py"],
         outputs=["data/interim/ntm_by_type.csv",
                  "data/interim/ntm_sector.csv"],
         requires=["data/raw/ntm/NTM-Trade-Frequency-Coverage-Ratio.csv",
                   "data/interim/ntm_country.csv",
                   "selection/importers_vn.csv",
                   "selection/eu_tariff_mapping.csv"],
         minutes=0.2,
         note="Đọc ntm_country.csv của bước 13; thiếu nó thì script thoát ngay."))

run(Step("22c_ntm_ave", "NTM quy về tương đương thuế (%)",
         ["scripts/build_ntm_ave.py"],
         outputs=["data/interim/ntm_ave_vn.csv"],
         requires=["data/raw/ntm/ave_gtap/UNCTADGTAP11_AVEborder.csv"],
         minutes=0.2,
         note="File AVE của UNCTAD-GTAP không tải được bằng script; nó đi kèm raw/."))

nbtools.describe("data/interim/ntm6_observed.csv", n=5)

### Bước 23. Biến kiểm soát, Green LPI, và thuế đối ứng Mỹ theo năm

**23a — `build_covariates.py`** chạy bảy module liên tiếp, mỗi module một bảng
khoá theo `(importer, year)` để bước 24 ghép thẳng:

1. **FTA (DESTA)** — hiệp định nào có hiệu lực với Việt Nam, từ năm nào. Đây là
   phép vá cho vấn đề mã nhóm ở bước 6.
2. **TTBD** — vụ kiện chống bán phá giá / trợ cấp / tự vệ.
3. **Gravity (CEPII)** — khoảng cách và các biến bất biến theo thời gian.
4. **Cú sốc chung** — World Bank Pink Sheet, chỉ số bất định chính sách EPU.
5. **Vĩ mô (WDI)** → `macro_panel_v2.csv`.
6. **Độ phức tạp (Atlas)** — PCI và ECI.
7. **Thuế đối ứng Mỹ 2025** → `us_tariffs_2025.csv` (mức thô theo dòng thuế).

**23b — `build_glpi.py`** đọc `macro_panel_v2.csv` + EPI và dựng **bốn** biến thể
Green LPI đã công bố. Không có một cách dựng duy nhất trong tài liệu, nên cả bốn
đều được tính và **chọn cái nào là một quyết định nghiên cứu**, không phải quyết
định kỹ thuật — panel mang cả bốn cột.

**23c — `build_us_tariff_panel.py`** biến `us_tariffs_2025.csv` thành bảng
`(importer, year)` mà merge ghép được: bốn cách đọc năm 2025 của Việt Nam, và
việc chọn cái nào cũng là quyết định của nhóm.

| Cột | Việt Nam 2025 |
|---|---|
| `us_recip_rate_yearend` | mức đang có hiệu lực ngày 31/12 — **20** |
| `us_recip_rate_peak` | mức cao nhất từng được ấn định — **46** |
| `us_recip_rate_days_wt` | trung bình theo số ngày có hiệu lực trong năm |
| `us_recip_rate_terminated` | 1 nếu dòng thuế đỉnh đã bị bãi |

`us_recip_rate_days_wt` là cách tóm tắt trung thực cho một mức thuế chỉ có hiệu
lực một phần của năm, và nó **nhỏ hơn nhiều** cả 46 lẫn 20 — đó chính là điểm
đáng nói. Mô hình đọc 46 (hay cả 20) cho toàn bộ năm 2025 là phóng đại mức phơi
nhiễm của năm đó lên hơn hai lần.

> ⚠️ **23c là bước đã từng bị bỏ quên, và nó không hề báo lỗi khi bị quên.**
> `merge_panel.py` bỏ qua trong im lặng mọi bảng phụ không tìm thấy
> (`if not table: continue`), nên thiếu `us_tariff_vn.csv` thì panel vẫn dựng
> xong — chỉ là **không có cột nào về cú sốc Mỹ 2025**, đúng cú sốc mà thiết kế
> nghiên cứu lấy làm trung tâm. Đó là lý do mọi bước từ đây khai `requires`.

In [ ]:
run(Step("23a_covariates", "7 module biến kiểm soát (DESTA, TTBD, CEPII, Pink Sheet, WDI, Atlas, US)",
         ["scripts/build_covariates.py"],
         outputs=["data/interim/fta_vn.csv", "data/interim/ttbd_vn.csv",
                  "data/interim/gravity_vn.csv", "data/interim/shocks_annual.csv",
                  "data/interim/macro_panel_v2.csv",
                  "data/interim/complexity_product.csv",
                  "data/interim/complexity_country.csv",
                  "data/interim/us_tariffs_2025.csv"],
         requires=["data/raw/rta", "data/raw/ttbd", "data/raw/gravity",
                   "data/raw/shocks", "data/raw/wdi", "data/raw/complexity",
                   "data/raw/us_tariffs_2025"],
         minutes=0.6))

run(Step("23b_glpi", "Green LPI, 4 biến thể đã công bố",
         ["scripts/build_glpi.py"],
         outputs=["data/interim/glpi.csv"],
         requires=["data/interim/macro_panel_v2.csv",
                   "data/raw/epi/epi2026results.xlsx"],
         minutes=0.2,
         note="Đọc macro_panel_v2.csv nên phải chạy sau 23a."))

run(Step("23c_us_tariff_panel", "Thuế đối ứng Mỹ 2025 -> bảng (importer, year)",
         ["scripts/build_us_tariff_panel.py"],
         outputs=["data/interim/us_tariff_vn.csv",
                  "data/interim/us_tariff_2025_monthly.csv"],
         requires=["data/interim/us_tariffs_2025.csv"],
         minutes=0.2,
         note="Thiếu bước này thì merge im lặng bỏ hết cột us_recip_* — xem phần chữ."))

nbtools.describe("data/interim/glpi.csv", n=5, show_columns=12)

nbtools.md("**Bốn cách đọc năm 2025 của Việt Nam (bước 23c):**")
nbtools.describe("data/interim/us_tariff_vn.csv", n=5, show_columns=12)

nbtools.md("**TTBD dừng ở 2015 — `ttbd_observed` là cờ, đừng đọc 0 là 'không có vụ kiện':**")
nbtools.frame(nbtools.value_counts("data/interim/ttbd_vn.csv", "year", 5))

---
## Phần 3 — Merge thành data tổng (bước 24–26)

Đây là chỗ mọi module gặp nhau. `merge_panel.py` lấy `episodes.csv` làm khung và
gắn từng bảng lên, mỗi bảng theo **khoá của riêng nó** — không phải cái gì cũng
ghép được bằng `(importer, year)`.

| Module | Ghép bằng khoá |
|---|---|
| thuế TRAINS | (reporter thuế, family, năm) — EU đọc `EUN` |
| EVFTA, CBAM, miễn trừ Mỹ | (family) hoặc (family, năm) |
| vĩ mô, gravity, FTA, TTBD, GLPI, thuế Mỹ 2025 | (importer, năm) |
| NTM6 | (nước áp, family, năm khảo sát) |
| độ phức tạp | (family) cho PCI, (importer, năm) cho ECI |

**Nguyên tắc xuyên suốt: không điền giá trị một cách âm thầm.** Mỗi bảng kéo
sang năm khác đều mang theo một cột `*_source_year`, nên luôn tách được dòng đo
đúng năm của nó với dòng mượn. Hai trần kéo khác nhau vì chúng trả lời hai câu
khác nhau: `MAX_CARRY_FORWARD = 3` cho thuế (một mức thuế 4 năm cũ có thể đơn
giản là sai), `MAX_GRAVITY_CARRY = 4` cho gravity (khoảng cách địa lý không đổi).

⚠️ **Một bảng phụ thiếu thì merge không báo lỗi** — với sáu bảng khoá
`(importer, year)` nó chỉ `continue` và đi tiếp, cho ra một panel thiếu cột mà
không ai biết. `requires` của bước 24 dưới đây liệt kê đúng những file
`merge_panel.py` thật sự mở, nên nếu bước nào phía trên bị bỏ sót, notebook
dừng ở đây với tên file còn thiếu thay vì giao ra một panel khuyết.

(Hai ngoại lệ có báo lỗi hẳn hoi, vì chúng là khung chứ không phải bảng phụ:
thiếu `episodes.csv` hay `ntm_country.csv` thì merge thoát ngay.)

Đo được: **4 phút 15, đỉnh RAM 378 MB** — nhẹ vì nó chia `episodes.csv` thành
shard và xử lý từng mảnh.

### Bước 24. Chạy merge

In [ ]:
run(Step("24_merge", "Gắn toàn bộ module lên episodes -> panel_final.csv",
         ["scripts/merge_panel.py"],
         outputs=["data/interim/panel_final.csv"],
         requires=[
             # khung
             "data/interim/episodes.csv",
             "selection/eu_tariff_mapping.csv",
             "data/interim/family_map.csv",
             # thuế: bảng EU đã ghép + biểu TRAINS thô cho 120 nước còn lại
             "data/interim/eu_tariff_panel.csv",
             "data/raw/tariffs/mfn",
             "data/raw/tariffs/pref",
             # NTM — ba cách đo, cả ba đều được đọc
             "data/interim/ntm6_observed.csv",
             "data/interim/ntm_country.csv",
             "data/interim/ntm_sector.csv",
             "data/interim/ntm_ave_vn.csv",
             # cú sốc chính sách
             "data/interim/cbam_products.csv",
             "data/interim/us_exempt_products.csv",
             "data/interim/us_tariff_vn.csv",
             # biến kiểm soát
             "data/interim/macro_panel_v2.csv",
             "data/interim/fta_vn.csv",
             "data/interim/ttbd_vn.csv",
             "data/interim/gravity_vn.csv",
             "data/interim/shocks_annual.csv",
             "data/interim/glpi.csv",
             "data/interim/complexity_product.csv",
             "data/interim/complexity_country.csv",
         ],
         minutes=2.1,
         note="Bản v2 đo 4'15 với đỉnh RAM 378 MB; lượt dựng lại 22/09/2026 mất 2'06. "
              "Xử lý theo shard nên nhẹ RAM."))

### Bước 25. Đọc `panel_final.csv` — data tổng

Đây chính là cái bạn gọi là "data tổng": mọi biến thô và mọi biến cùng-năm đã có
mặt, nhưng **chưa có độ trễ và chưa có biến cần group-by**. Đó là việc của
bước 27.

Grain: một dòng = một `(spell_id, year)`.

In [ ]:
nbtools.describe("data/interim/panel_final.csv", n=3)
nbtools.unique_key("data/interim/panel_final.csv", ["spell_id", "year"])

nbtools.md("**171 cột, đọc theo nhóm chủ đề:**")
nbtools.column_groups("data/interim/panel_final.csv", {
    "định danh + target": ("spell_id", "importer", "exporter", "product_family",
                            "year", "event", "t_start", "t_stop", "censor", "trunc"),
    "thương mại": ("import_value", "net_weight", "unit_value", "rca", "share",
                    "hhi", "growth", "total_import"),
    "thuế": ("tariff", "pref_margin", "evfta", "staging"),
    "NTM": ("ntm",),
    "cú sốc Mỹ / CBAM": ("us_recip", "us_transship", "cbam"),
    "vĩ mô + gravity": ("gdp", "population", "inflation", "exchange", "lpi",
                         "glpi", "dist", "contig", "comlang", "wto", "rta"),
    "phòng vệ thương mại": ("ad_", "cvd_", "sg_", "ttb"),
    "giá hàng hoá / bất định": ("price_", "cmo_", "gepu"),
})

### Bước 26. Kiểm tra sự trung thực: `*_source_year`

Cell quan trọng nhất của Phần 3, và là cell dễ bị bỏ qua nhất.

Một giá trị nằm trong ô không có nghĩa là nó được **đo** ở năm đó. Các cột
`*_source_year` cho biết nó đến từ năm nào. Quy tắc đọc:

* `source_year == year` → đo đúng năm đó.
* `source_year < year` → kéo từ năm trước sang. Chấp nhận được trong trần đã
  đặt, nhưng phải kể ra trong phần hạn chế.
* `source_year > year` → **lấy từ tương lai**. Chỉ được phép ở biến được ghi
  nhận là tĩnh. Đây đúng là lỗi F6 của v1: NTM6 35,1%, `ntm_ave` 47,6%,
  `ntm_survey_year` 40,6%, LPI 9,7% số dòng mang giá trị tương lai.

Cell dưới đếm trực tiếp ba nhóm đó trên panel đang có. Nó gom mọi phép đếm vào
**một** lượt quét: `panel_final.csv` nặng ~850 MB, quét lại một lượt cho mỗi cột
thì mất vài phút mà không được gì thêm.

In [ ]:
import polars as pl

path = nbtools.abs_path("data/interim/panel_final.csv")
sy_cols = [c for c in nbtools.columns(path) if c.endswith("_source_year")]
nbtools.md("**Các cột đánh dấu năm nguồn:** " + ", ".join(f"`{c}`" for c in sy_cols))

lf = pl.scan_csv(path, infer_schema_length=20_000, ignore_errors=True)
aggs = []
for c in sy_cols:
    sy = pl.col(c).cast(pl.Int32, strict=False)
    aggs += [
        pl.col(c).is_null().sum().alias(f"{c}|null"),
        (sy == pl.col("year")).sum().alias(f"{c}|same"),
        (sy < pl.col("year")).sum().alias(f"{c}|past"),
        (sy > pl.col("year")).sum().alias(f"{c}|future"),
    ]
got = lf.select(pl.len().alias("n"), *aggs).collect().row(0, named=True)

n = got["n"]
nbtools.table(
    [[c] + [f"{got[f'{c}|{k}'] / n:.1%}" for k in ("same", "past", "future", "null")]
     for c in sy_cols],
    ["Cột", "Đo đúng năm", "Kéo từ quá khứ", "⚠️ Từ tương lai", "Null"],
)
nbtools.note("Cột 'Từ tương lai' phải bằng 0% ở mọi biến biến-thiên-theo-thời-gian. "
             "Giá trị khác 0 chỉ chấp nhận được ở biến đã ghi nhận là tĩnh "
             "(ví dụ `ntm_ave` — một ước lượng 2017 cho mỗi nước).")

---
## Phần 4 — Panel Stage 1 (bước 27–29)

`panel_final.csv` còn thiếu hai thứ trước khi mô hình chạy được:

**1. Năm biến B3 cần group-by trên chính panel** (không phải join cột):
`volatility_3y`, `n_products_to_c`, `n_markets_for_p`, `hs2_share`,
`log_total_import_cp`.

**2. Độ trễ t−1 của mọi biến vào mô hình nguy cơ.** B3 nói rõ: "mọi covariate
lấy tại t−1" — dùng giá trị cùng năm sẽ tạo đồng thời tính, vì một quan hệ sắp
chết thì kim ngạch đã sụp trước đó rồi.

**Tạo độ trễ ở đây KHÔNG phải là `.shift(1)`.** Một cặp (importer, family) có thể
mang nhiều spell cách nhau nhiều năm thật (kết thúc 2009, mở lại 2014), và ngay
trong một spell, `GAP_TOLERANCE = 1` đã chèn một năm dưới ngưỡng thành dòng riêng.
`.shift(1)` sau khi sắp xếp sẽ âm thầm kéo nhầm giá trị qua cả hai loại khe. Mọi
độ trễ ở đây là **join theo năm dương lịch thật**, ở đúng grain mà biến đó sống:

| Grain của biến | Join theo |
|---|---|
| mức quan hệ (kim ngạch, thị phần, thuế của chính nước đó) | (importer, family, year−1) |
| mức nước-năm (GDP, dân số, độ rộng danh mục) | (importer, year−1), trên bảng đã khử trùng lặp |
| mức sản phẩm-năm (RCA, tăng trưởng của mặt hàng, cầu thế giới) | (family, year−1) |
| chính sách EU (thuế, biên ưu đãi, mức cắt EVFTA) | đọc thẳng từ `eu_tariff_panel.csv` |
| mức importer × HS2-năm (`hs2_share`) | (importer, hs2, year−1) |

Đây chính là chỗ lỗi F1 nằm: v1 lấy độ trễ của `log_total_import_cp` ở grain
(family, year) bằng một `.unique()` tuỳ tiện, trong khi nó là tổng nhập khẩu
**của từng nước** — **sai trên 94,6% số dòng**, và không tất định giữa hai lần
chạy.

⚠️ **Bước 27 và 28 phải chạy tách rời.** Đừng gộp thành `build_stage1_df.py all`
trên máy dưới ~6 GB RAM trống — lý do nằm trong docstring đầu
[`scripts/build_stage1_df.py`](scripts/build_stage1_df.py). Bước 27 ghi ra một
parquet tạm và bước 28 xoá nó sau khi ghép xong, nên `done_if` của bước 27 trỏ
vào panel cuối: đã có panel thì không dựng lại file tạm cho vô ích.

### Bước 27. `features` — biến group-by và toàn bộ độ trễ

In [ ]:
run(Step("27_features", "Dựng biến B3 + toàn bộ độ trễ t-1",
         ["scripts/build_stage1_df.py", "features"],
         outputs=["data/final/_features_tmp.parquet"],
         done_if=["data/final/stage1_panel.parquet"],
         requires=["data/interim/panel_final.csv",
                   "data/interim/eu_tariff_panel.csv",
                   "selection/eu_tariff_mapping.csv"],
         minutes=0.4,
         note="Phải chạy TÁCH RỜI với bước 28 vì RAM."))

### Bước 28. `join` — ghép lại và ghi panel cuối

In [ ]:
run(Step("28_join", "Ghép feature vào panel -> stage1_panel.parquet",
         ["scripts/build_stage1_df.py", "join"],
         outputs=["data/final/stage1_panel.parquet"],
         requires=["data/final/_features_tmp.parquet",
                   "data/interim/panel_final.csv"],
         minutes=0.6,
         note="Nén zstd. Kết quả: 932.204 dòng × 209 cột, ~178 MB."))

### Bước 29. Đọc panel cuối

Đích đến. 209 cột, và cách duy nhất để đọc hết là đi theo từ điển dữ liệu:
[`docs/TU_DIEN_DU_LIEU_FINAL_DF.md`](docs/TU_DIEN_DU_LIEU_FINAL_DF.md) — nguồn,
grain và công thức cho từng cột.

Cell này in ra cấu trúc, phân bố target, và mẫu B0 (EU27, 2012–2024) mà Stage 2
nên nhắm tới: **146.042 dòng / 15.958 sự kiện**.

In [ ]:
import polars as pl

nbtools.describe("data/final/stage1_panel.parquet", n=3)

nbtools.md("**209 cột theo nhóm:**")
nbtools.column_groups("data/final/stage1_panel.parquet", {
    "định danh + target": ("spell_id", "importer", "exporter", "product_family",
                            "year", "event", "t_start", "t_stop", "censor",
                            "trunc", "right_censored", "gap_filled"),
    "độ trễ t-1 (_lag)": ("_lag",),
    "thuế + EVFTA": ("tariff", "pref_margin", "evfta", "staging"),
    "NTM": ("ntm",),
    "Mỹ 2025 + CBAM": ("us_recip", "us_transship", "cbam"),
    "vĩ mô + gravity + GLPI": ("gdp", "population", "inflation", "exchange",
                                "lpi", "glpi", "dist", "contig", "comlang"),
})

lf = pl.scan_parquet(nbtools.abs_path("data/final/stage1_panel.parquet"))
nbtools.md("**Target:**")
nbtools.frame(lf.select("event").group_by("event").len().sort("event").collect())

nbtools.md("**Mẫu B0 — EU27, 2012-2024, chỉ dòng 'sống' (mục tiêu của Stage 2):**")
EU27 = ["AUT","BEL","BGR","HRV","CYP","CZE","DNK","EST","FIN","FRA","DEU","GRC",
        "HUN","IRL","ITA","LVA","LTU","LUX","MLT","NLD","POL","PRT","ROU","SVK",
        "SVN","ESP","SWE"]
# gap_filled == 0: loại các dòng-năm dưới ngưỡng mà GAP_TOLERANCE chèn vào giữa
# spell. Đây đúng là bộ lọc audit_stage1.py dùng cho B0.
b0 = lf.filter(pl.col("importer").is_in(EU27)
               & (pl.col("gap_filled") == 0)
               & pl.col("year").is_between(2012, 2024))
nbtools.frame(b0.select(
    pl.len().alias("dòng"),
    pl.col("event").sum().alias("sự kiện"),
    pl.col("spell_id").n_unique().alias("spell"),
    pl.col("product_family").n_unique().alias("family"),
).collect())
nbtools.note("Phải khớp con số đã chốt của bản v2: **146.042 dòng / 15.958 sự kiện**. "
             "Bỏ `gap_filled == 0` ra thì được 152.894 dòng — đó là lý do bộ lọc này "
             "phải viết đúng.")

---
## Phần 5 — Kiểm định và bàn giao (bước 30–32)

### Bước 30. 18 phép thử

`audit_stage1.py` **tính lại** mọi phép thử từ parquet và, ở chỗ nào cần, từ
chính file raw — không bao giờ đọc lại một cột mà bản dựng tự khai về mình. Đó
là toàn bộ ý nghĩa của bộ kiểm: một bản dựng chỉ được quyền nói là đã sửa khi có
một phép đọc độc lập đồng ý.

Nó đọc parquet theo từng nhóm cột để chạy được trên máy 8 GB, và ghi ra
`docs/audit/stage1_local.md` cùng một file JSON số liệu thô.

**Cả 18 phép phải pass.** Chính script này **fail 11 phép trên panel v1** — đó là
cách ta biết bộ kiểm có cắn thật, chứ không phải 18 phép thử dễ dãi.

Một phép trong số đó **không chạy được bằng một lượt**: A18 — *hai lần dựng liên
tiếp phải cho file giống hệt nhau ở mọi cột* — cần `--compare <parquet thứ hai>`.
Không có cờ đó, báo cáo ghi thẳng `A18 | n/a | not run`, và 17 phép còn lại vẫn
chạy. Bước 30b ngay dưới lo phần A18, và nó **tắt sẵn** vì phải dựng panel lần
thứ hai.

> **Hai chi tiết nhỏ, cùng một lý do: báo cáo bạn đọc phải là báo cáo của bản
> bạn vừa dựng.**
>
> `--tag local` → ghi ra `docs/audit/stage1_local.md`, **không** đụng vào
> `stage1_v2.md`. File `_v2` là báo cáo tham chiếu của bản v2; giữ nguyên nó thì
> mới so được hai bên.
>
> `force=True` → luôn chạy lại, kể cả khi báo cáo đã có. Nếu không, quy tắc "bỏ
> qua khi output đã có" sẽ cho bạn xem một báo cáo cũ (hoặc báo cáo người gửi
> kèm theo) thay vì kết quả thật của máy bạn — tức là đọc một lời tự khai, đúng
> thứ mà bộ kiểm sinh ra để không phải tin. Nó chỉ mất 23 giây.

In [ ]:
run(Step("30_audit", "Chạy 18 phép kiểm định độc lập",
         ["scripts/audit_stage1.py", "--tag", "local",
          "--family-map", "data/interim/family_map.csv"],
         outputs=["docs/audit/stage1_local.md", "docs/audit/stage1_local.json"],
         requires=["data/final/stage1_panel.parquet",
                   "data/interim/family_map.csv",
                   "data/raw/trade", "data/raw/tariffs/mfn"],
         minutes=0.5,
         note="Luôn chạy lại (force=True) — xem phần chữ."),
    force=True)

### Bước 30b. A18 — dựng lại lần hai và so từng cột (tuỳ chọn, tắt sẵn)

A18 là phép thử duy nhất cần **hai** bản panel. `build_stage1_df.py` ghi cứng
`data/final/stage1_panel.parquet`, nên muốn có hai bản để so thì phải dọn chỗ:
đổi tên bản thứ nhất, dựng lại bước 27 + 28, rồi gọi `audit_stage1.py --compare`.

Cell dưới làm đúng ba việc đó, và **trả bản thứ nhất về chỗ cũ nếu lượt dựng lại
hỏng** — nên một lần chạy dở dang không làm bạn mất panel.

Bật bằng cách sửa `RUN_A18 = True`. Trên máy tham chiếu, bước 27 + 28 + kiểm
định mất khoảng **1,5 phút**; trên máy chậm hơn hoặc ít RAM hơn thì lâu hơn
đáng kể, nên nó không nằm trong lượt Run All thường.

> Kết quả mong đợi: `same_columns: true`, `columns_differing: []`. Đã kiểm ngày
> 22/09/2026: một bản dựng lại **chỉ từ `data/raw/`, hoàn toàn ngoại tuyến** cho
> ra panel trùng với bản v2 tham chiếu ở **mọi cột**, và cả 39 bảng trung gian
> trong `data/interim/` đều **giống hệt từng byte**.

In [ ]:
import json

RUN_A18 = False

panel = nbtools.abs_path("data/final/stage1_panel.parquet")
first = nbtools.abs_path("data/final/stage1_panel_run1.parquet")

if not RUN_A18:
    nbtools.note("`RUN_A18 = False` — bỏ qua. Sửa thành `True` rồi chạy lại cell này "
                 "nếu muốn tự kiểm A18.")
elif first.exists():
    nbtools.fail(f"Đã có `{nbtools.rel(first)}` từ một lượt trước. Tự quyết giữ bản "
                 "nào (nó là một bản dựng hoàn chỉnh), xoá/đổi tên đi rồi chạy lại.")
elif not panel.exists():
    nbtools.fail("Chưa có panel để so — chạy bước 27 và 28 trước.")
else:
    panel.rename(first)
    try:
        run(nbtools.REGISTRY["27_features"], force=True)
        run(nbtools.REGISTRY["28_join"], force=True)
        run(Step("30b_a18", "A18: so hai bản dựng liên tiếp, từng cột",
                 ["scripts/audit_stage1.py", "--tag", "local_a18",
                  "--family-map", "data/interim/family_map.csv",
                  "--compare", nbtools.rel(first)],
                 outputs=["docs/audit/stage1_local_a18.md",
                          "docs/audit/stage1_local_a18.json"],
                 requires=["data/final/stage1_panel.parquet"],
                 minutes=0.5), force=True)
        a18 = json.loads(nbtools.abs_path("docs/audit/stage1_local_a18.json")
                         .read_text(encoding="utf-8"))["checks"]["A18_determinism"]
        nbtools.kv(a18, headers=("A18", "Kết quả"))
        if a18 and a18.get("same_columns") and not a18.get("columns_differing"):
            nbtools.ok("A18 pass — hai lượt dựng cho cùng một panel ở mọi cột.")
        else:
            nbtools.fail("A18 FAIL — có cột lệch giữa hai lượt dựng. Đừng dùng panel này.")
    finally:
        # Dựng lại hỏng giữa đường -> trả bản thứ nhất về chỗ cũ, đừng để trống.
        if panel.exists():
            first.unlink()
        else:
            first.rename(panel)
            nbtools.warn("Lượt dựng lại không ra file; đã trả bản thứ nhất về "
                         f"`{nbtools.rel(panel)}`.")

### Bước 31. Đọc báo cáo kiểm định

In [ ]:
import json

rep = json.loads(nbtools.abs_path("docs/audit/stage1_local.json")
                 .read_text(encoding="utf-8"))

nbtools.md("**Tóm tắt panel theo bộ kiểm (tính lại độc lập từ parquet):**")
s = rep["summary"]
nbtools.kv({
    "Dòng": f"{s['rows']:,}",
    "Spell": f"{s['spells']:,}",
    "Nước nhập khẩu": f"{s['importers']:,}",
    "Family": f"{s['families']:,}",
    "Sự kiện": f"{s['events']:,}",
    "B0 (EU27, 2012-2024, dòng sống)": f"{s['b0_rows']:,} dòng / {s['b0_events']:,} sự kiện",
})

nbtools.md("**Giá trị thô của từng phép thử** (cột Pass nằm trong báo cáo .md bên dưới):")
nbtools.table(
    [[k, json.dumps(v, ensure_ascii=False)[:110]] for k, v in rep["checks"].items()],
    ["Phép thử", "Giá trị"],
)

nbtools.md("---")
nbtools.md(nbtools.abs_path("docs/audit/stage1_local.md").read_text(encoding="utf-8"))

### Bước 32. Bàn giao

Panel đã xong. Nạp nó vào phân tích chỉ cần ba dòng:

```python
import polars as pl
panel = pl.scan_parquet("data/final/stage1_panel.parquet")
b0 = panel.filter(pl.col("importer").is_in(EU27) & pl.col("year").is_between(2012, 2024))
```

Dùng `scan_parquet` chứ đừng `read_parquet`: panel 178 MB nén nhưng bung ra RAM
thì lớn hơn nhiều, và hầu hết phân tích chỉ cần vài chục cột.

**Cần CSV thay vì parquet** (Excel, vài bản R/Stata cũ):
```python
pl.scan_parquet("data/final/stage1_panel.parquet").sink_csv("data/final/stage1_panel.csv")
```

### Những giới hạn phải mang theo vào phần Thảo luận

Không phải lỗi — là ranh giới đã biết, đã đo, và đã ghi. Đừng để chúng bị phát
hiện lại ở vòng phản biện:

1. **Ưu đãi theo mã nhóm chưa lấy** (F2, hoãn theo D3). `tariff_rate` của MYS,
   SGP, THA, IDN, PHL, KHM, BRN, IND, NZL, CAN, MEX, PER, HKG là **100% MFN**.
   Phân tích thuế ngoài EU chưa dùng được.
2. **TTBD dừng ở 2015Q4.** Từ 2016 là null, không phải 0. Đọc `ttbd_observed`
   trước khi đọc `ttb_any_in_force`.
3. **CEPII Gravity dừng ở 2021.** Các cột biến thiên theo thời gian trong bảng
   gravity (`entry_*`, `wto_d`, `fta_wto`) không nên đọc cho 2022–2025.
4. **NTM: 95,2% độ phủ nhưng chỉ 36,6% đo đúng năm.** Luôn đọc kèm
   `ntm6_source_year`.
5. **2025 có 0 sự kiện**, theo đúng thiết kế: dưới quy tắc khe hở, một cái chết ở
   năm D chỉ xác nhận được khi đã thấy D+1 và D+2.
6. **Green LPI có bốn biến thể, và cả bốn đều bất biến theo thời gian**, vì
   `build_glpi.py` đọc EPI ở dạng lát cắt ngang. Bước 12 đã đặt sẵn nguyên liệu
   (`epi_indicators_annual.csv`) để sửa; gộp nó thành điểm EPI theo năm là một
   quyết định nghiên cứu chưa chốt.
7. **CBAM trong cửa sổ panel là chi phí tuân thủ, không phải giá** — giai đoạn
   chuyển tiếp chỉ yêu cầu báo cáo, chế độ chính thức bắt đầu 01/01/2026.
8. **Thuế Mỹ 2025 có bốn cách đọc** (`yearend` 20, `peak` 46, `days_wt`,
   `terminated`). Chọn cái nào là quyết định của nhóm; đọc 46 cho cả năm là
   phóng đại.
9. **30% dòng bị phơi ra ở các lần đổi đời HS vẫn bị kiểm duyệt**, không giải
   được bằng gộp trong cùng HS4 (D1).

### Đọc tiếp

| | |
|---|---|
| Từng cột nghĩa là gì | [`docs/TU_DIEN_DU_LIEU_FINAL_DF.md`](docs/TU_DIEN_DU_LIEU_FINAL_DF.md) |
| Khoá ghép giữa các module | [`docs/KHOA_GHEP_STAGE1_PANEL.md`](docs/KHOA_GHEP_STAGE1_PANEL.md) |
| Mọi lỗi v1, cách sửa, và §7 phần còn tồn | [`docs/STAGE1_PANEL_FIX_PLAN.md`](docs/STAGE1_PANEL_FIX_PLAN.md) |
| Thu thập gì từ đâu, phủ tới đâu | [`docs/DATA_HANDOFF.md`](docs/DATA_HANDOFF.md) |
| Khung nghiên cứu B0–B9 của thầy hướng dẫn | [`Stage1_Research_Framework.md`](Stage1_Research_Framework.md) |
| Cài đặt, sự cố, cách gửi folder này đi tiếp | [`README.md`](README.md) |

> Bốn link `docs/…` ở trên chỉ mở được nếu bản bạn nhận có kèm folder `docs/`.
> Nó thường bị bỏ ra cho gọn (nó không cần để chạy), và **bước 1 đã nói rõ bản
> này có hay không**. Thiếu thì xin người gửi — riêng `docs/audit/` thì bước 30
> tự tạo lại.

### Bảng kê toàn bộ pipeline

`plan()` không tham số kê **đúng những bước mà các cell ở trên đã định nghĩa**,
theo thứ tự chúng xuất hiện — không phải một danh sách chép tay song song, vốn là
thứ sớm muộn cũng lệch khỏi notebook. Nếu bảng dưới ngắn hơn bạn nghĩ thì có
cell phía trên chưa chạy: bấm **Run All**.

In [ ]:
plan()